In [ ]:
# Homework 12: https://almondine-song-c43.notion.site/Homework-12-1cc34d22fd9a430bb2b210cb9ce3db71

In [19]:
# ====================================
# GROTH16 PROVER
# ====================================
# The prover generates a zero-knowledge proof that demonstrates knowledge of a valid witness
# for an R1CS circuit without revealing the private witness values.
#
# High-level steps:
# 1. Convert R1CS to QAP (Quadratic Arithmetic Program) using Lagrange interpolation
# 2. Compute witness polynomials U(x), V(x), W(x) from the QAP and witness
# 3. Calculate quotient polynomial h(x) = (U(x)·V(x) - W(x)) / t(x)
# 4. Generate proof components (A, B, C, X) using trusted setup parameters
# 5. Add zero-knowledge randomness to ensure the proof reveals nothing about the witness

import galois
import numpy as np
import random
from functools import reduce
from py_ecc.bn128 import G1, G2, add, multiply, curve_order, pairing, neg

GF = galois.GF(curve_order, primitive_element=5, verify=False)

# langrange interpolation
def interpolate_column(col):
    xs = GF(np.arange(1, len(col) + 1)) # x-vector
    return galois.lagrange_poly(xs, col) # interpolating y-vector through points in col

# innter product over multiple polynomials (multiply each polynomial with witness coefficient)
def inner_product_polynomials_with_witness(polys, witness):
    mul_ = lambda x, y: x * y
    sum_ = lambda x, y: x + y
    return reduce(sum_, map(mul_, polys, witness))

# generate structured reference string (srs) over G
def generate_srs(discrete_log, G, n):
    powers = [pow(discrete_log, i, curve_order) for i in range(n)]
    return [multiply(G, power) for power in powers]

# multiply polynomial coeffs with srs
def mul_poly_with_srs(poly_coeffs, point_vector):
    mul_ = lambda x, y: multiply(x, y)
    sum_ = lambda x, y: add(x, y)
    return reduce(sum_, map(mul_, point_vector, poly_coeffs))

def eval_polynomials(polys, discrete_log):
    return np.array([poly(discrete_log) for poly in polys])

def generate_proof(R1CS, a):

    # Extract R1CS matrices
    A = R1CS[0]
    B = R1CS[1]
    C = R1CS[2]

    # Convert R1CS to Galois field elements
    L_galois = GF((A + curve_order) % curve_order)
    R_galois = GF((B + curve_order) % curve_order)
    O_galois = GF((C + curve_order) % curve_order)
    a_galois = GF(a)

    # Lagrange interpolation
    U_polys = np.apply_along_axis(interpolate_column, 0, L_galois)
    V_polys = np.apply_along_axis(interpolate_column, 0, R_galois)
    W_polys = np.apply_along_axis(interpolate_column, 0, O_galois)

    # Inner product with witness
    Ua = inner_product_polynomials_with_witness(U_polys, a_galois)
    Va = inner_product_polynomials_with_witness(V_polys, a_galois)
    Wa = inner_product_polynomials_with_witness(W_polys, a_galois)

    # Generate helper polynomials h(x) and t(x)
    # ===================================================================

    # Calculate t
    t = (
        galois.Poly([1, curve_order-1], field=GF)
        * galois.Poly([1, curve_order-2], field=GF)
        * galois.Poly([1, curve_order-3], field=GF)
    )

    # Calculate h
    h = (Ua * Va - Wa) // t

    # Structured reference string (SRS)
    # ===================================================================

    # Generate random elements for the SRS
    tau = random.randint(1,1000) % curve_order
    alpha = random.randint(1,1000) % curve_order
    beta = random.randint(1,1000) % curve_order
    gamma =  random.randint(1,1000) % curve_order
    gamma_inv = pow(gamma, -1, curve_order) % curve_order
    delta =  random.randint(1,1000) % curve_order
    delta_inv = pow(delta, -1, curve_order) % curve_order

    r =  random.randint(1,1000) % curve_order
    s =  random.randint(1,1000) % curve_order

    srs_G1 = generate_srs(tau,G1,3)
    srs_G2 = generate_srs(tau,G2,3)

    t_tau = t(GF(tau)) * delta_inv
    srs_HT = [multiply(ec_point, int(t_tau)) for ec_point in generate_srs(tau,G1, 3)]

    # Generate proof
    # ===================================================================
    
    # Precompute constant terms
    Alpha1 = multiply(G1, alpha)
    Beta1 = multiply(G1, beta)
    Beta2 = multiply(G2, beta)
    Gamma2 = multiply(G2, gamma)
    Delta1 = multiply(G1, delta)
    Delta2 = multiply(G2, delta)

    # Compute proof elements
    poly_coeffs = [int(coeff) for coeff in Ua.coefficients(size=L_galois.shape[0],order="asc")]
    A1 = add(add(mul_poly_with_srs(poly_coeffs, srs_G1), Alpha1), multiply(Delta1, r))

    poly_coeffs = [int(coeff) for coeff in Va.coefficients(size=R_galois.shape[0],order="asc")]
    B1 = add(add(mul_poly_with_srs(poly_coeffs, srs_G1), Beta1), multiply(Delta1, s))
    B2 = add(add(mul_poly_with_srs(poly_coeffs, srs_G2), Beta2), multiply(Delta2, s))

    poly_coeffs = [int(coeff) for coeff in h.coefficients(size=O_galois.shape[0],order="asc")]
    HT = mul_poly_with_srs(poly_coeffs, srs_HT) # divide by delta
    
    # Compute Eps polynomial evaluations
    Eps = alpha * eval_polynomials(V_polys, tau) + beta * eval_polynomials(U_polys,tau) + eval_polynomials(W_polys,tau)

    # Split Eps into public and private parts
    l = 3
    a_pub = a_galois[:l]
    a_priv = a_galois[l:]

    Eps_pub = Eps[:l]
    Eps_priv = Eps[l:]

    Eps_pub_a = inner_product_polynomials_with_witness(Eps_pub, a_pub) * gamma_inv 
    Eps_priv_a = inner_product_polynomials_with_witness(Eps_priv, a_priv) * delta_inv

    Eps_pub1 = multiply(G1, int(Eps_pub_a))
    Eps_priv1 = multiply(G1, int(Eps_priv_a))

    # Compute final public and private proof elements
    X1 = Eps_pub1
    C1 = add(add(add(add(Eps_priv1,HT),multiply(A1,s)),multiply(B1,r)),multiply(Delta1,(curve_order - r*s % curve_order) % curve_order ))

    print("Proof successful.")

    return (A1, B2, C1, X1, Alpha1, Beta2, Gamma2, Delta2)

In [20]:
# ====================================
# PROBLEM STATEMENT & WITNESS GENERATION
# ====================================
# We're proving knowledge of a solution to the arithmetic circuit:
# out = 3x²y + 5xy - x - 2y + 3
#
# Without revealing the private values of x, y, and intermediate computations.
# The witness vector contains: [1, out, x, y, v1, v2]
# where v1 = 3x² and v2 = v1·y are intermediate values.
#
# The R1CS matrices encode the constraints:
# Constraint 1: 3x · x = v1
# Constraint 2: v1 · y = v2  
# Constraint 3: v2 + 5xy - x - 2y + 3 = out

# Define the matrices
A = np.array([[0,0,3,0,0,0],
               [0,0,0,0,1,0],
               [0,0,1,0,0,0]])

B = np.array([[0,0,1,0,0,0],
               [0,0,0,1,0,0],
               [0,0,0,5,0,0]])

C = np.array([[0,0,0,0,1,0],
               [0,0,0,0,0,1],
               [-3,1,1,2,0,-1]])

# pick values for x and y
x = GF(random.randint(1,1000))
y = GF(random.randint(1,1000))

# this is our orignal formula: 3 * x * x * y + 5 * x * y + (-1 * x) + (-2 * y) + 3

# result given chosen witness in GF
out = GF(3) * x * x * y + GF(5) * x * y + GF(-1 + curve_order)* x + GF(-2 + curve_order) * y + GF(3)

# intermediate values
v1 = GF(3) * x * x
v2 = v1 * y

# witness vector
a = np.array([1, out, x, y, v1, v2])

A1, B2, C1, X1, Alpha, Beta, Gamma, Delta = generate_proof([A,B,C], a)


Proof successful.


In [21]:
# ====================================
# GROTH16 VERIFIER
# ====================================
# The verifier checks the proof without knowing the private witness values
# using the pairing equation:
# e(A, B) = e(α, β) · e(X, γ) · e(C, δ)
#
# Where:
# - e() is the bilinear pairing function
# - A, B, C are proof components from the prover
# - X is the public input component
# - α, β, γ, δ are from the verification key (trusted setup)

from py_ecc.bn128.bn128_curve import FQ12
from py_ecc.bn128 import eq, neg, pairing, final_exponentiate

# Groth16 verification equation
# We check if: e(A, -B) · e(α, β) · e(γ, X) · e(δ, C) = 1
# Note: Using -B is equivalent to checking e(A,B) = e(α,β)·e(γ,X)·e(δ,C)
verification_result = eq(
    FQ12.one(),  # Target value (identity element)
    final_exponentiate(
        pairing(B2, neg(A1))      # e(B, -A) - proof components
        * pairing(Beta, Alpha)     # e(β, α) - verification key check
        * pairing(Gamma, X1)       # e(γ, X) - public input check
        * pairing(Delta, C1)       # e(δ, C) - witness validity check
    )
)

print("Verification returns: ", verification_result)  # Returns True if proof is valid, False otherwise

Verification returns:  True
